# Translation Tool

A small interactive translator built with Python widgets and the open-source LibreTranslate service. Choose languages, enter text, and translate without leaving the notebook.

Judy Mohamed Abidou

In [3]:
import html
import json
import requests
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript

LIBRETRANSLATE_MIRRORS = [
    "https://translate.argosopentech.com",
    "https://libretranslate.de",
    "https://translate.mentality.rip",
    "https://translate.api.skitzen.com",
    "https://trans.zillyhuhn.com",
    "https://libretranslate.pussthecat.org",
]
LIBRETRANSLATE_URL = None
MYMEMORY_URL = "https://api.mymemory.translated.net/get"

FALLBACK_LANGUAGES = [
    ("English", "en"), ("Arabic", "ar"), ("Spanish", "es"),
    ("French", "fr"), ("German", "de"), ("Italian", "it"),
    ("Portuguese", "pt"), ("Russian", "ru"), ("Japanese", "ja"),
    ("Korean", "ko"), ("Chinese", "zh"), ("Hindi", "hi"),
    ("Turkish", "tr"), ("Dutch", "nl"), ("Swedish", "sv"),
    ("Polish", "pl"), ("Ukrainian", "uk"), ("Indonesian", "id"),
    ("Vietnamese", "vi"), ("Thai", "th"), ("Persian", "fa"),
    ("Urdu", "ur"), ("Hebrew", "he"),
]
LANGUAGES = [("Detect language", "auto")] + FALLBACK_LANGUAGES
LANGUAGE_NAMES = {code: name for name, code in LANGUAGES}
translation_state = {"text": "", "source": None, "target": None, "detected": None}

def split_utf8_text(text, max_bytes=450):
    """Split text into pieces below MyMemory's documented 500-byte input limit."""
    chunks = []
    remaining = text
    while remaining:
        byte_count = 0
        cutoff = 0
        for index, character in enumerate(remaining):
            size = len(character.encode("utf-8"))
            if byte_count + size > max_bytes:
                break
            byte_count += size
            cutoff = index + 1
        if cutoff == 0:
            cutoff = 1
        if cutoff < len(remaining):
            whitespace = [i for i, char in enumerate(remaining[:cutoff]) if char.isspace()]
            if whitespace and whitespace[-1] > cutoff // 2:
                cutoff = whitespace[-1] + 1
        chunks.append(remaining[:cutoff])
        remaining = remaining[cutoff:]
    return chunks

def translate_with_mymemory(text, source, target):
    source_code = "autodetect" if source == "auto" else source
    translated_chunks = []
    detected = source
    for chunk in split_utf8_text(text):
        response = requests.get(
            MYMEMORY_URL,
            params={"q": chunk, "langpair": f"{source_code}|{target}"},
            timeout=15,
        )
        data = response.json()
        if not response.ok or str(data.get("responseStatus")) != "200":
            detail = data.get("responseDetails") or f"HTTP {response.status_code}"
            raise RuntimeError(str(detail))
        response_data = data.get("responseData") or {}
        translated_chunk = response_data.get("translatedText")
        if not translated_chunk:
            raise RuntimeError("MyMemory returned an empty translation.")
        translated_chunks.append(html.unescape(translated_chunk))
        detected = response_data.get("detectedLanguage", detected)
    return "".join(translated_chunks), detected

def translate_text(text, source, target):
    """Try MyMemory, then fall back to the public LibreTranslate mirrors."""
    global LIBRETRANSLATE_URL
    errors = []
    try:
        return translate_with_mymemory(text, source, target)
    except Exception as exc:
        errors.append(f"MyMemory: {exc}")
    candidates = ([LIBRETRANSLATE_URL] if LIBRETRANSLATE_URL else [])
    candidates += [mirror for mirror in LIBRETRANSLATE_MIRRORS if mirror != LIBRETRANSLATE_URL]
    payload = {"q": text, "source": source, "target": target, "format": "text"}
    for mirror in candidates:
        try:
            response = requests.post(mirror.rstrip("/") + "/translate", json=payload, timeout=10)
            data = response.json()
            if response.ok and data.get("translatedText"):
                LIBRETRANSLATE_URL = mirror
                detected = (data.get("detectedLanguage") or {}).get("language", source)
                return data["translatedText"], detected
            message = data.get("error") or f"HTTP {response.status_code}"
            errors.append(f"{mirror}: {message}")
        except Exception as exc:
            errors.append(f"{mirror}: {exc}")
    raise RuntimeError("MyMemory and the public LibreTranslate mirrors were unreachable or rejected this request. Check Kaggle Internet and daily/provider limits. Details: " + "; ".join(errors[-3:]))

print("No API key or Google billing is needed. MyMemory will be tried first; LibreTranslate mirrors are backups.")

No API key or Google billing is needed. MyMemory will be tried first; LibreTranslate mirrors are backups.


In [4]:
source_language = widgets.Dropdown(
    options=LANGUAGES, value="auto", description="",
    layout=widgets.Layout(width="100%"),
)
source_language.add_class("lingua-select")
target_options = [item for item in LANGUAGES if item[1] != "auto"]
target_default = "ar" if any(code == "ar" for _, code in target_options) else target_options[0][1]
target_language = widgets.Dropdown(
    options=target_options, value=target_default, description="",
    layout=widgets.Layout(width="100%"),
)
target_language.add_class("lingua-select")
source_text = widgets.Textarea(
    placeholder="Type or paste something to translate…",
    layout=widgets.Layout(width="100%", height="190px"),
)
translated_output = widgets.HTML(
    value='<div style="color:#aab5b0">Your translation will appear here</div>',
    layout=widgets.Layout(width="100%", min_height="190px"),
)
source_text.add_class("lingua-input")
detected_output = widgets.HTML(value="")
error_output = widgets.HTML(value="")
status_output = widgets.HTML(
    value=(
        '<span class="lingua-status">✿ Free service · no key needed</span>'
    )
)

swap_button = widgets.Button(description="↔", tooltip="Swap languages", layout=widgets.Layout(width="46px", height="42px"), style=widgets.ButtonStyle(button_color="#f6e8eb"))
translate_button = widgets.Button(description="Translate", layout=widgets.Layout(width="138px", height="44px"), style=widgets.ButtonStyle(button_color="#8a526e"))
clear_button = widgets.Button(description="Clear", layout=widgets.Layout(width="78px", height="38px"), style=widgets.ButtonStyle(button_color="#fff8f5"))
copy_button = widgets.Button(description="Copy", layout=widgets.Layout(width="78px", height="38px"), style=widgets.ButtonStyle(button_color="#fff8f5"))
speak_source_button = widgets.Button(description="Listen", layout=widgets.Layout(width="88px", height="36px"), style=widgets.ButtonStyle(button_color="#f8edf0"))
speak_result_button = widgets.Button(description="Listen", layout=widgets.Layout(width="88px", height="36px"), style=widgets.ButtonStyle(button_color="#f8edf0"))
swap_button.add_class("lingua-swap")
translate_button.add_class("lingua-primary")
clear_button.add_class("lingua-quiet")
copy_button.add_class("lingua-quiet")
speak_source_button.add_class("lingua-listen")
speak_result_button.add_class("lingua-listen")
feedback_output = widgets.Output()

header = widgets.HTML(
    '<div class="lingua-header">'
    '<div class="lingua-brand"><span class="lingua-flower">✿</span><span>lingua</span><small>LANGUAGE NOTES</small></div>'
    '<div class="lingua-kicker">YOUR LITTLE TRANSLATION STUDIO</div>'
    '<h1>Say it with <em>feeling.</em></h1>'
    '<p>For all the words you wish you could say.</p>'
    '<div class="lingua-ornament">✦ &nbsp; ✿ &nbsp; ✦</div></div>'
)
from_label = widgets.HTML('<div class="lingua-caption">FROM</div>')
to_label = widgets.HTML('<div class="lingua-caption">TO</div>')
source_panel = widgets.VBox([widgets.HTML('<div class="lingua-panel-title">Your words</div>'), source_text, speak_source_button], layout=widgets.Layout(width="48%", padding="17px"))
result_panel = widgets.VBox([widgets.HTML('<div class="lingua-panel-title">A little translation</div>'), translated_output, detected_output, speak_result_button], layout=widgets.Layout(width="48%", padding="17px"))
source_panel.add_class("lingua-panel")
result_panel.add_class("lingua-panel")
language_row = widgets.HBox(
    [widgets.VBox([from_label, source_language], layout=widgets.Layout(width="44%")),
     swap_button,
     widgets.VBox([to_label, target_language], layout=widgets.Layout(width="44%"))],
    layout=widgets.Layout(align_items="center", justify_content="space-between", width="100%")
)
language_row.add_class("lingua-language-row")
panels = widgets.HBox(
    [source_panel, result_panel],
    layout=widgets.Layout(justify_content="space-between", width="100%")
)
panels.add_class("lingua-panels")
action_row = widgets.HBox([clear_button, copy_button, translate_button], layout=widgets.Layout(align_items="center", justify_content="space-between", width="100%"))
action_row.add_class("lingua-actions")
app = widgets.VBox(
    [header, language_row, panels, error_output, action_row, status_output, feedback_output],
    layout=widgets.Layout(max_width="980px", padding="26px")
)
app.add_class("lingua-app")

display(HTML("""
<style>
.lingua-app { max-width:980px; margin:10px auto; padding:26px; background:linear-gradient(145deg,#fffaf7 0%,#fffdfb 58%,#fbf7fc 100%); border:1px solid #efdee2; border-radius:24px; box-shadow:0 18px 48px rgba(108,70,89,.09); color:#493945; font-family:'Trebuchet MS','Segoe UI',sans-serif; }
.lingua-app * { box-sizing:border-box; }
.lingua-header { text-align:center; padding:2px 8px 20px; border-bottom:1px solid #f0e3e5; margin-bottom:4px; }
.lingua-brand { display:flex; align-items:center; gap:7px; color:#67485b; font-family:Georgia,serif; font-size:23px; letter-spacing:-.7px; font-weight:700; }
.lingua-brand small { margin-left:5px; color:#a9929d; font:700 9px 'Trebuchet MS',sans-serif; letter-spacing:1.5px; }
.lingua-flower { color:#c4788c; font-size:22px; }
.lingua-kicker { margin-top:17px; color:#aa8291; font-size:10px; font-weight:700; letter-spacing:2px; }
.lingua-header h1 { margin:8px 0 4px; color:#493544; font:500 36px/1.2 Georgia,'Times New Roman',serif; letter-spacing:-.8px; }
.lingua-header h1 em { color:#a95f79; font-weight:400; }
.lingua-header p { margin:0; color:#8d7d87; font:14px/1.6 Georgia,serif; }
.lingua-ornament { margin-top:12px; color:#c996a4; font-size:11px; letter-spacing:3px; }
.lingua-caption { margin:5px 0 4px; color:#a47c8c; font:700 10px 'Trebuchet MS',sans-serif; letter-spacing:1.5px; }
.lingua-language-row { padding:16px 0 17px; border-bottom:1px solid #f0e3e5; }
.lingua-app .widget-dropdown select { min-height:40px; padding:7px 12px; border:1px solid #ead9df; border-radius:11px; background:#fffdfc; color:#55424f; font:14px 'Trebuchet MS','Segoe UI',sans-serif; box-shadow:0 2px 7px rgba(108,70,89,.035); }
.lingua-app .lingua-panel { border:1px solid #efdee2; border-radius:17px; background:rgba(255,255,255,.78); box-shadow:0 5px 16px rgba(108,70,89,.045); }
.lingua-app .lingua-panel-title { padding-bottom:8px; color:#654b5b; font:600 15px Georgia,'Times New Roman',serif; }
.lingua-app .widget-textarea textarea { padding:13px 14px; border:1px solid #eadde0; border-radius:12px; outline:none; background:#fffdfc; color:#473943; font:15px/1.65 Georgia,'Times New Roman',serif; box-shadow:inset 0 1px 3px rgba(108,70,89,.035); }
.lingua-app .widget-textarea textarea:focus { border-color:#c58c9d; box-shadow:0 0 0 3px rgba(197,140,157,.13); }
.lingua-app .widget-textarea textarea::placeholder { color:#b5a5ac; }
.lingua-app .widget-button button { border:1px solid #ecd9df!important; border-radius:12px!important; background:#fff8f6!important; color:#79576a!important; font:600 12px 'Trebuchet MS','Segoe UI',sans-serif!important; box-shadow:0 3px 8px rgba(108,70,89,.06)!important; transition:transform .15s,box-shadow .15s!important; }
.lingua-app .widget-button button:hover { transform:translateY(-1px); box-shadow:0 6px 14px rgba(108,70,89,.11)!important; }
.lingua-app .lingua-primary button { border-color:#8a526e!important; background:linear-gradient(135deg,#a66881,#81516d)!important; color:#fff!important; font-size:13px!important; letter-spacing:.2px; }
.lingua-app .lingua-swap button { border-radius:50%!important; color:#945f77!important; font-size:20px!important; }
.lingua-app .lingua-status { display:inline-block; margin-top:5px; padding:8px 12px; border:1px solid #e9dce4; border-radius:999px; background:#fbf4f7; color:#806277; font:11px 'Trebuchet MS',sans-serif; }
.lingua-app .lingua-actions { padding-top:13px; border-top:1px solid #f0e3e5; }
.lingua-app .lingua-panels { padding:16px 0; }
.lingua-app .lingua-listen { margin-top:7px; }
@media (max-width:700px) { .lingua-app { padding:16px!important; } .lingua-header h1 { font-size:29px; } .lingua-app .lingua-panels { flex-direction:column!important; } .lingua-app .lingua-panel { width:100%!important; margin-bottom:12px; } .lingua-app .lingua-actions { flex-wrap:wrap; gap:8px; justify-content:flex-start!important; } .lingua-app .lingua-actions .lingua-primary { margin-left:auto; } }
</style>
"""))

def show_placeholder():
    translated_output.value = '<div style="color:#aab5b0">Your translation will appear here</div>'
    detected_output.value = ""

def on_translate(_):
    text = source_text.value.strip()
    error_output.value = ""
    if not text:
        error_output.value = '<div style="color:#a64d3c;padding:8px 0">Enter some text to translate.</div>'
        return
    if len(text) > 5000:
        error_output.value = '<div style="color:#a64d3c;padding:8px 0">Please keep your text under 5,000 characters.</div>'
        return
    if source_language.value == target_language.value:
        error_output.value = '<div style="color:#a64d3c;padding:8px 0">Choose two different languages.</div>'
        return
    translate_button.disabled = True
    translate_button.description = "Translating…"
    try:
        result, detected = translate_text(text, source_language.value, target_language.value)
        translated_output.value = (
            '<div dir="auto" style="min-height:150px;white-space:pre-wrap;overflow-wrap:anywhere;'
            'font-size:17px;line-height:1.7;color:#263b36;padding-top:12px">'
            + html.escape(result) + '</div>'
        )
        target_name = LANGUAGE_NAMES.get(target_language.value, target_language.value)
        detected_name = LANGUAGE_NAMES.get(detected, detected or "Unknown")
        detected_output.value = (
            f'<div style="font-size:11px;color:#84918b;padding:5px 0">'
            f'{"Detected: " + html.escape(detected_name) if source_language.value == "auto" else ""}'
            f'<span style="float:right">To {html.escape(target_name)}</span></div>'
        )
        translation_state.update(text=result, source=text, target=target_language.value, detected=detected)
    except Exception as exc:
        error_output.value = f'<div style="color:#a64d3c;padding:8px 0">{html.escape(str(exc))}</div>'
    finally:
        translate_button.disabled = False
        translate_button.description = "Translate  →"

def on_clear(_):
    source_text.value = ""
    translation_state.update(text="", source=None, target=None, detected=None)
    error_output.value = ""
    show_placeholder()

def on_swap(_):
    old_source, old_target = source_language.value, target_language.value
    detected = translation_state.get("detected")
    source_language.value = old_target
    available_codes = {code for _, code in LANGUAGES}
    target_language.value = detected if old_source == "auto" and detected in available_codes else ("en" if old_source == "auto" else old_source)
    if translation_state.get("text"):
        old_input = source_text.value
        source_text.value = translation_state["text"]
        translated_output.value = (
            '<div dir="auto" style="min-height:150px;white-space:pre-wrap;overflow-wrap:anywhere;'
            'font-size:17px;line-height:1.7;color:#263b36;padding-top:12px">'
            + html.escape(old_input) + '</div>'
        )
        translation_state.update(text=old_input, source=source_text.value, target=target_language.value, detected=None)
        detected_output.value = ""
    error_output.value = ""

def run_browser_js(script):
    with feedback_output:
        feedback_output.clear_output(wait=True)
        display(Javascript(script))

def on_copy(_):
    text = translation_state.get("text", "")
    if not text:
        error_output.value = '<div style="color:#a64d3c;padding:8px 0">Translate some text before copying.</div>'
        return
    run_browser_js("navigator.clipboard.writeText(" + json.dumps(text) + ").then(() => alert('Translation copied')).catch(() => alert('Clipboard access is unavailable in this browser.'));")

def on_speak(text, language):
    if not text:
        return
    run_browser_js(
        "window.speechSynthesis.cancel(); const u = new SpeechSynthesisUtterance("
        + json.dumps(text) + "); u.lang = " + json.dumps(language or "en") + "; window.speechSynthesis.speak(u);"
    )

translate_button.on_click(on_translate)
clear_button.on_click(on_clear)
swap_button.on_click(on_swap)
copy_button.on_click(on_copy)
speak_source_button.on_click(lambda _: on_speak(source_text.value, source_language.value if source_language.value != "auto" else "en"))
speak_result_button.on_click(lambda _: on_speak(translation_state.get("text"), target_language.value))
display(app)